# Test process_r.py — Local End-to-End Validation

This notebook tests the `input_transform_intensities` Python runner locally, **without AWS buckets or the MD platform**. It uses in-memory sample data to verify that:

1. The Python runner triggers the R workflow correctly
2. The R function runs and returns the expected output structure
3. Output has the required fields: `intensity`, `metadata`, `runtime_metadata`

**Prerequisites:** Run from the MDCustomR repo root. Install the Python package (`pip install -e .`) and the R package (`Rscript install.R` or `devtools::install()`).

In [1]:
import os
import sys

# Set working directory to repo root (parent of tutorial/)
# Run this notebook from MDCustomR root, or from tutorial/ — we adjust accordingly
cwd = os.getcwd()
repo_root = os.path.dirname(cwd) if os.path.basename(cwd) == "tutorial" else cwd
os.chdir(repo_root)
sys.path.insert(0, repo_root)

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/mansiaggarwal/Desktop/Main/Git_Repos/MDCustomR


In [2]:
import rpy2.robjects as ro
print(ro.r('.libPaths()'))

Error importing in API mode: ImportError("dlopen(/opt/anaconda3/envs/md_custom_r/lib/python3.11/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <587B6A46-0BED-3992-853A-486017AC0683> /opt/anaconda3/envs/md_custom_r/lib/python3.11/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


[1] "/opt/anaconda3/envs/md_custom_r/lib/R/library"



## 1. Create sample intensity and metadata dataframes

The `transformIntensities` R function expects:
- **Intensity:** `GroupId`, `replicate`, `NormalisedIntensity` (plus optional columns)
- **Metadata:** `GeneNames`, `GroupId`, `GroupLabel`, `GroupLabelType`, `ProteinIds`, `Description`

In [3]:
import pandas as pd
import numpy as np

# Minimal intensity data: 3 proteins x 3 replicates
# NormalisedIntensity: raw values (R will apply log2 before normalization)
intensity_data = pd.DataFrame({
    "GroupId": [1, 1, 1, 2, 2, 2, 3, 3, 3],
    "replicate": ["rep1", "rep2", "rep3"] * 3,
    "NormalisedIntensity": [100.0, 105.0, 98.0, 200.0, 210.0, 195.0, 50.0, 52.0, 48.0],
})

# Metadata for the same GroupIds
metadata_data = pd.DataFrame({
    "GroupId": [1, 2, 3],
    "GeneNames": ["GeneA", "GeneB", "GeneC"],
    "GroupLabel": ["Protein A", "Protein B", "Protein C"],
    "GroupLabelType": ["Protein", "Protein", "Protein"],
    "ProteinIds": ["P1", "P2", "P3"],
    "Description": ["", "", ""],
})

print("Intensity shape:", intensity_data.shape)
print("Metadata shape:", metadata_data.shape)
intensity_data

Intensity shape: (9, 3)
Metadata shape: (3, 6)


,GroupId,replicate,NormalisedIntensity
0,1,rep1,100.0
1,1,rep2,105.0
2,1,rep3,98.0
3,2,rep1,200.0
4,2,rep2,210.0
5,2,rep3,195.0
6,3,rep1,50.0
7,3,rep2,52.0
8,3,rep3,48.0


In [4]:
import uuid
from md_dataset.models.dataset import (
    InputDatasetTable,
    IntensityInputDataset,
    IntensityTableType,
    DatasetType,
)

# Table names must match what IntensityInputDataset.table() expects
# ("Protein_Intensity" and "Protein_Metadata")
intensity_table = InputDatasetTable(name="Protein_Intensity", data=intensity_data)
metadata_table = InputDatasetTable(name="Protein_Metadata", data=metadata_data)

input_dataset = IntensityInputDataset(
    id=uuid.uuid4(),
    name="Test Intensity Dataset",
    type=DatasetType.INTENSITY,
    tables=[intensity_table, metadata_table],
)

print(f"Input dataset: {input_dataset.name}")
print(f"Tables: {[t.name for t in input_dataset.tables]}")

Input dataset: Test Intensity Dataset
Tables: ['Protein_Intensity', 'Protein_Metadata']


## 3. Call the Python runner and execute R

The `input_transform_intensities` function returns `RFuncArgs` (arguments for R), not the R output. The `@md_r` decorator only runs R when the function is invoked inside a Prefect flow on the MD platform. Hence, to run the following code chunk, temporailiy comment the lines with the decorate (e.g., line 25 and 42 in `src/md_custom_r/process_r.py`).

In [5]:
# from src.md_custom_r.process_r import input_transform_intensities, MDCustomRParams
# from md_dataset.process import run_r_task

# params = MDCustomRParams(normalisation_methods="scale")

# # Step 1: Get RFuncArgs (arguments that would be passed to R on the platform)
# r_func_args = input_transform_intensities(
#     input_datasets=[input_dataset],
#     params=params,
#     output_dataset_type=DatasetType.INTENSITY,
# )

# # Step 2: Execute R locally to get actual workflow output
# output = run_r_task(
#     r_file="./src/md_custom_r/process.R",
#     r_function="run_transform_intensities",
#     r_preparation=r_func_args,
# )

from src.md_custom_r.process_r import run_transform_intensities, MDCustomRParams
from md_dataset.models.dataset import DatasetType

params = MDCustomRParams(normalisation_methods="scale")

output = run_transform_intensities(
    input_datasets=[input_dataset],
    params=params,
    output_dataset_type=DatasetType.INTENSITY,
)

/opt/anaconda3/envs/md_custom_r/lib/python3.11/site-packages/pydantic_settings/main.py:447: UserWarning: Config key `pyproject_toml_table_header` is set in model_config but will be ignored because no PyprojectTomlConfigSettingsSource source is configured. To use this config key, add a PyprojectTomlConfigSettingsSource source to the settings sources via the settings_customise_sources hook.
  cls._settings_warn_unused_config_keys(sources, cls.model_config)
/opt/anaconda3/envs/md_custom_r/lib/python3.11/site-packages/pydantic_settings/main.py:447: UserWarning: Config key `toml_file` is set in model_config but will be ignored because no TomlConfigSettingsSource source is configured. To use this config key, add a TomlConfigSettingsSource source to the settings sources via the settings_customise_sources hook.
  cls._settings_warn_unused_config_keys(sources, cls.model_config)


ImportError: cannot import name 'run_transform_intensities' from 'src.md_custom_r.process_r' (/Users/mansiaggarwal/Desktop/Main/Git_Repos/MDCustomR/src/md_custom_r/process_r.py)

## 2. Build IntensityInputDataset (in-memory, no AWS)

Use `InputDatasetTable` with `data=` to pass dataframes directly — no `bucket` or `key` needed.

## 2. Build IntensityInputDataset (in-memory, no AWS)

Use `InputDatasetTable` with `data=` to pass dataframes directly — no `bucket` or `key` needed.

## 4. Verify RFuncArgs input and run R

First verify the Python runner produced correct `RFuncArgs` (data passed to R). `r_func_args.data_frames[0]` = Protein_Intensity, `r_func_args.data_frames[1]` = metadata. Then run R via `run_r_task` and verify the output.

In [ ]:
# Verify RFuncArgs: data_frames[0] = Protein_Intensity, data_frames[1] = metadata
assert len(r_func_args.data_frames) == 2, "RFuncArgs should have 2 data_frames"

print("r_func_args.data_frames[0] is Protein_Intensity (intensity input):")
display(r_func_args.data_frames[0].head())

print("r_func_args.data_frames[1] is metadata:")
display(r_func_args.data_frames[1].head())

In [ ]:
# Verify R output has required fields (output comes from run_r_task)
required = ["intensity", "metadata"]
optional = ["runtime_metadata"]

assert "intensity" in output, "Missing required 'intensity' in output"
assert "metadata" in output, "Missing required 'metadata' in output"

print("✓ Output has required fields: intensity, metadata")

if "runtime_metadata" in output:
    print("✓ Output has runtime_metadata")
    print("Runtime metadata:")
    display(output["runtime_metadata"])

print("\nIntensity output shape:", output["intensity"].shape)
print("Metadata output shape:", output["metadata"].shape)

In [ ]:
# Show transformed intensity (NormalisedIntensity updated by R workflow)
output["intensity"]

In [ ]:
print("Test passed: process_r.py ran successfully and returned correct output structure.")